# DuckDB Fundamentals Explorer

Notebook for exploring `pystocks` V2 fundamentals data.

Data sources:
- DuckDB views: `/Users/alex/Documents/pystocks/data/fundamentals/fundamentals.duckdb`
- Parquet partitions: `/Users/alex/Documents/pystocks/data/fundamentals/parquet`
- Event index: `/Users/alex/Documents/pystocks/data/fundamentals/events.db`


In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

ROOT = Path('/Users/alex/Documents/pystocks')
DUCKDB_PATH = ROOT / 'data' / 'fundamentals' / 'fundamentals.duckdb'
assert DUCKDB_PATH.exists(), f'Missing DuckDB file: {DUCKDB_PATH}'

con = duckdb.connect(str(DUCKDB_PATH))
print('Connected:', DUCKDB_PATH)


## 1) What objects exist?

In [ ]:
con.sql('SHOW TABLES').df()

## 2) Endpoint coverage and effective date ranges

In [ ]:
endpoint_catalog = con.sql('SELECT * FROM endpoint_catalog ORDER BY endpoint').df()
endpoint_catalog

## 3) Recent events (metadata only)

In [ ]:
recent_events = con.sql('''
SELECT
  conid,
  endpoint,
  effective_at,
  observed_at,
  effective_source,
  payload_hash,
  payload_size_raw,
  payload_size_compressed
FROM endpoint_events_all
ORDER BY observed_at DESC
LIMIT 50
''').df()
recent_events

## 4) Validate irregular time-series spacing per endpoint

In [ ]:
timeseries_profile = con.sql('''
WITH base AS (
  SELECT conid, endpoint, CAST(effective_at AS DATE) AS d
  FROM endpoint_events_all
), gaps AS (
  SELECT
    conid,
    endpoint,
    d,
    d - LAG(d) OVER (PARTITION BY conid, endpoint ORDER BY d) AS gap_days
  FROM base
)
SELECT
  endpoint,
  COUNT(*) AS n_points,
  COUNT(DISTINCT conid) AS n_conids,
  MIN(d) AS min_effective_date,
  MAX(d) AS max_effective_date,
  AVG(gap_days) AS avg_gap_days
FROM gaps
GROUP BY endpoint
ORDER BY endpoint
''').df()
timeseries_profile

## 5) Inspect dynamic columns for a specific endpoint view

`f__...` columns are flattened scalar fields pulled from raw endpoint payloads.


In [ ]:
# Change this endpoint slug as needed
slug = 'ratios'
view_name = f'endpoint_{slug}'

cols = con.sql(f"DESCRIBE {view_name}").df()
cols

In [ ]:
# Pull a compact sample for wrangling in pandas
sample_ratios = con.sql('''
SELECT *
FROM endpoint_ratios
ORDER BY effective_at DESC
LIMIT 200
''').df()

sample_ratios.head()

## 6) Find columns that are mostly null (schema sparsity check)

In [ ]:
def null_profile(df: pd.DataFrame, top_n: int = 40) -> pd.DataFrame:
    out = pd.DataFrame({
        'column': df.columns,
        'null_pct': [float(df[c].isna().mean()) for c in df.columns],
        'n_unique': [int(df[c].nunique(dropna=True)) for c in df.columns],
    }).sort_values(['null_pct', 'n_unique'], ascending=[False, True])
    return out.head(top_n)

null_profile(sample_ratios, top_n=50)

## 7) Query Parquet directly (optional)

Useful if you want to bypass views and inspect raw partition files.

In [ ]:
PARQUET_GLOB = '/Users/alex/Documents/pystocks/data/fundamentals/parquet/endpoint=*/year=*/month=*/*.parquet'

con.sql(f"SELECT endpoint, COUNT(*) AS n FROM read_parquet('{PARQUET_GLOB}', union_by_name=true) GROUP BY endpoint ORDER BY endpoint").df()

In [ ]:
# Optional cleanup when done with notebook session
con.close()